# Memory Lab — Agent Memory Systems Benchmark (Google Colab)

Notebook này benchmark **Memory Lab** (`VectorMemory`, `GraphMemory`, `ZeroTokenMemory`, `HybridMemory`) trực tiếp với các hệ thống/baseline phổ biến khác trên cùng tập dữ liệu, cùng embedding model và cùng độ đo chuẩn IR:

- **Memory Lab Strategies**: `MemoryLab-ZeroToken`, `MemoryLab-Graph`, `MemoryLab-Vector`, `MemoryLab-Hybrid`
- **External / Reference Systems**:
  - `Naive-Recency` (sliding window theo thời gian gần nhất)
  - `RankBM25` (Okapi BM25 chuẩn từ thư viện `rank-bm25`)
  - `FAISS-FlatIP` (tìm kiếm vector cosine chính xác bằng FAISS)
  - `ChromaDB-Dense` (vector store ChromaDB in-memory)
  - `Mem0-OSS` *(tùy chọn, bật `ENABLE_MEM0 = True` nếu có `OPENAI_API_KEY`)*
- **Độ đo**: `Recall@1`, `Recall@3`, `Recall@5`, `MRR@5`, `nDCG@5`, độ trễ `Ingest/Query (p50, p95 ms)`, và số lần gọi Embedding/LLM.
- **Phân tích theo nhóm câu hỏi**: `factual_recall`, `multi_hop_graph`, `temporal_reasoning`, `exact_identifier`, `bilingual_vi_en`.

> **Mẹo chạy trên Colab**: Chọn **Runtime → Change runtime type → T4 GPU** (hoặc CPU đều chạy tốt với `BAAI/bge-small-en-v1.5` hoặc `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`).


## 1. Cài đặt môi trường & Clone Memory Lab


In [ ]:
#@title Cài đặt dependencies & clone repository { display-mode: "form" }
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/npv2k1/memory-lab.git"  #@param {type:"string"}
REPO_BRANCH = "feat/complete-memory-lab"  #@param {type:"string"}
INSTALL_DEPS = True  #@param {type:"boolean"}
ENABLE_MEM0 = False  #@param {type:"boolean"}

REPO_DIR = Path("/content/memory-lab") if Path("/content").exists() else Path.cwd()

if not (REPO_DIR / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Using existing repository at {REPO_DIR}")

if INSTALL_DEPS:
    pkgs = [
        "sentence-transformers>=3.0",
        "rank-bm25>=0.2.2",
        "faiss-cpu>=1.8.0",
        "chromadb>=0.5.0",
        "pandas>=2.2",
        "matplotlib>=3.8",
        "numpy>=1.26",
    ]
    if ENABLE_MEM0:
        pkgs.append("mem0ai>=0.1.0")
    if sys.version_info >= (3, 12):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    else:
        print(f"Detected Python {sys.version_info.major}.{sys.version_info.minor}; installing core runtime deps directly.")
        pkgs.extend(["fastapi>=0.115,<1", "pydantic-settings>=2.7,<3", "httpx>=0.28,<1", "qdrant-client>=1.14,<1.15"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

print("Setup complete! Repo:", REPO_DIR)


## 2. Cấu hình Embedding Provider (Miễn phí trên Colab hoặc OpenAI-compatible)

Mặc định dùng `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (hỗ trợ cả tiếng Anh và tiếng Việt, chạy trực tiếp trên Colab không tốn phí API). Bạn cũng có thể đổi sang `BAAI/bge-small-en-v1.5` hoặc trỏ sang OpenAI endpoint.


In [ ]:
#@title Chọn Embedding Model & cấu hình Benchmark { display-mode: "form" }
EMBEDDING_BACKEND = "sentence_transformers"  #@param ["sentence_transformers", "openai_compatible", "hash_offline"]
ST_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  #@param {type:"string"}
OPENAI_BASE_URL = "https://api.openai.com/v1"  #@param {type:"string"}
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"  #@param {type:"string"}
TOP_K = 5  #@param {type:"integer"}

import hashlib
import math
import re
from typing import Protocol


class CountingEmbeddingProvider:
    """Wraps any embedding backend and tracks exact API calls and embedded text counts."""

    def __init__(self, backend: str = "sentence_transformers"):
        self.backend = backend
        self.api_calls = 0
        self.texts_embedded = 0
        self._model = None
        self._openai = None

        if backend == "sentence_transformers":
            from sentence_transformers import SentenceTransformer
            self._model = SentenceTransformer(ST_MODEL_NAME)
            self.identity = f"st:{ST_MODEL_NAME}"
        elif backend == "openai_compatible":
            from memory_lab.retrieval.embedding import OpenAIEmbeddings
            api_key = os.environ.get("OPENAI_API_KEY", "")
            if not api_key:
                raise ValueError("Set os.environ['OPENAI_API_KEY'] before using openai_compatible backend")
            self._openai = OpenAIEmbeddings(OPENAI_BASE_URL, api_key, OPENAI_EMBEDDING_MODEL)
            self.identity = f"openai:{OPENAI_EMBEDDING_MODEL}"
        else:
            self.identity = "hash-offline-384"

    def reset_counters(self) -> None:
        self.api_calls = 0
        self.texts_embedded = 0

    def embed(self, texts: list[str]) -> list[list[float]]:
        self.api_calls += 1
        self.texts_embedded += len(texts)
        if self.backend == "sentence_transformers":
            vecs = self._model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
            return [v.tolist() for v in vecs]
        if self.backend == "openai_compatible":
            return self._openai.embed(texts)
        # Deterministic hashed bag-of-ngrams fallback for offline smoke testing
        dim = 256
        out = []
        for text in texts:
            vec = [0.0] * dim
            for tok in re.findall(r"[\w]+", text.lower()):
                h = int(hashlib.sha256(tok.encode()).hexdigest()[:8], 16)
                vec[h % dim] += 1.0
            norm = math.sqrt(sum(x * x for x in vec)) or 1.0
            out.append([x / norm for x in vec])
        return out


embedder = CountingEmbeddingProvider(EMBEDDING_BACKEND)
sample_vec = embedder.embed(["Memory Lab benchmark sanity check"])[0]
embedder.reset_counters()
print(f"Loaded embedding backend: {embedder.identity} (dim={len(sample_vec)})")


## 3. Bộ dữ liệu Benchmark đa chiều (Agent Long-Term Memory)

Bộ dữ liệu gồm **24 interaction traces** trải dài nhiều phiên làm việc và nhiều mốc thời gian (`2026-08` đến `2026-11`), cùng **20 truy vấn đánh giá** chia thành 5 nhóm năng lực cốt lõi:

1. `factual_recall`: Nhớ sự kiện & quyết định kỹ thuật cụ thể
2. `multi_hop_graph`: Suy luận nối thực thể qua quan hệ (`A -> B -> C`)
3. `temporal_reasoning`: Truy vấn theo mốc thời gian (`YYYY-MM`, `YYYY-MM-DD`)
4. `exact_identifier`: Mã lỗi, cờ cấu hình, commit/tag chính xác
5. `bilingual_vi_en`: Truy vấn tiếng Việt (có/không dấu) đối chiếu nội dung song ngữ

> Bạn có thể thay `BENCHMARK_MEMORIES` và `BENCHMARK_QUERIES` bằng tập dữ liệu của riêng mình (ví dụ export từ **LoCoMo** hoặc **LongMemEval**).


In [ ]:
import json
from memory_lab.storage.filesystem.store import digest

BENCHMARK_MEMORIES = [
    # --- Architecture & Vector/Graph decisions ---
    {"key": "m01", "session_id": "arch-01", "timestamp": "2026-08-12T09:15:00Z", "tags": ["architecture", "qdrant"],
     "content": "Memory Lab uses Qdrant as its rebuildable dense vector index with cosine similarity and payload filters."},
    {"key": "m02", "session_id": "arch-01", "timestamp": "2026-08-12T09:30:00Z", "tags": ["storage", "filesystem"],
     "content": "Filesystem artifacts are the immutable source of truth; every chunk preserves block_ids and raw source provenance."},
    {"key": "m03", "session_id": "arch-02", "timestamp": "2026-09-03T14:00:00Z", "tags": ["graph", "neo4j"],
     "content": "GraphMemory uses Neo4j for entity relation traversal and supports a zero-dependency FileGraphStore locally."},
    {"key": "m04", "session_id": "arch-02", "timestamp": "2026-09-04T11:20:00Z", "tags": ["graph", "redis"],
     "content": "Neo4j connects to Redis worker queues for asynchronous graph enrichment jobs in production deployments."},
    {"key": "m05", "session_id": "arch-03", "timestamp": "2026-09-18T16:45:00Z", "tags": ["sparse", "bm25"],
     "content": "ZeroTokenMemory relies on Okapi BM25 lexical ranking, entity overlap, and TemporalHierarchy with zero API token cost."},
    {"key": "m06", "session_id": "arch-03", "timestamp": "2026-09-19T10:10:00Z", "tags": ["hybrid", "rrf"],
     "content": "HybridMemory combines VectorMemory, GraphMemory, BM25 sparse search, and temporal recency via Reciprocal Rank Fusion (RRF)."},

    # --- Multi-hop entity chains ---
    {"key": "m07", "session_id": "team-01", "timestamp": "2026-10-02T08:00:00Z", "tags": ["team", "project"],
     "content": "Linh Nguyen authored OrionParser to convert scanned Vietnamese PDF invoices into structured tables."},
    {"key": "m08", "session_id": "team-01", "timestamp": "2026-10-02T08:40:00Z", "tags": ["parser", "docling"],
     "content": "OrionParser extends Docling with custom OCR post-processing and bounding-box normalization."},
    {"key": "m09", "session_id": "team-02", "timestamp": "2026-10-08T13:15:00Z", "tags": ["security", "gateway"],
     "content": "Minh Tran built SentinelProxy to enforce rate limits and mTLS authentication in front of FastAPI."},
    {"key": "m10", "session_id": "team-02", "timestamp": "2026-10-09T15:00:00Z", "tags": ["gateway", "nginx"],
     "content": "SentinelProxy integrates with Nginx upstream healthchecks on port 4321 for zero-downtime blue-green rollouts."},

    # --- Exact identifiers, incidents, error codes ---
    {"key": "m11", "session_id": "ops-01", "timestamp": "2026-10-14T22:10:00Z", "tags": ["incident", "qdrant"],
     "content": "Incident ERR-QDR-4091 occurred when collection dimension mismatch (768 vs 384) blocked upsert on namespace ns_9a4f2c."},
    {"key": "m12", "session_id": "ops-01", "timestamp": "2026-10-15T07:30:00Z", "tags": ["fix", "namespace"],
     "content": "Fix for ERR-QDR-4091: isolate Qdrant collections per embedding_namespace hash and rebuild from runs/*/vectors.json."},
    {"key": "m13", "session_id": "ops-02", "timestamp": "2026-10-21T17:05:00Z", "tags": ["ci", "docker"],
     "content": "Deployment tag sha-7175082 was promoted after Playwright E2E passed with E2E_BASE_URL=http://localhost:4321."},
    {"key": "m14", "session_id": "ops-02", "timestamp": "2026-10-22T09:00:00Z", "tags": ["config", "env"],
     "content": "Set MAX_UPLOAD_BYTES=52428800 and BUN_INSTALL=$HOME/.bun when provisioning Bun 1.4.4 on cloud devboxes."},

    # --- Temporal timeline & milestones ---
    {"key": "m15", "session_id": "sprint-10", "timestamp": "2026-10-28T10:00:00Z", "tags": ["milestone", "mcp"],
     "content": "On 2026-10-28 we finalized the MCP JSON-RPC 2.0 specification exposing memory_store, memory_search, memory_get, and memory_delete."},
    {"key": "m16", "session_id": "sprint-11", "timestamp": "2026-11-05T11:30:00Z", "tags": ["milestone", "eval"],
     "content": "On 2026-11-05 we added Recall@K, MRR, and nDCG@K evaluation metrics with deterministic dataset_hash snapshots."},
    {"key": "m17", "session_id": "sprint-11", "timestamp": "2026-11-12T15:20:00Z", "tags": ["cache", "pipeline"],
     "content": "On 2026-11-12 stage cache invalidation API POST /runs/{id}/invalidate was added to selectively re-run parse or index stages."},
    {"key": "m18", "session_id": "sprint-12", "timestamp": "2026-11-25T08:00:00Z", "tags": ["release", "ui"],
     "content": "On 2026-11-25 Memory Lab v0.2.0 launched the 3-tab Astro workbench: Documents, Memory Playground, and Experimentation Comparison UI."},

    # --- Bilingual Vietnamese / English memories ---
    {"key": "m19", "session_id": "vi-01", "timestamp": "2026-11-10T09:45:00Z", "tags": ["vietnamese", "enrichment"],
     "content": "Hệ thống Enrichment trích xuất thực thể, quan hệ và mốc thời gian hoàn toàn không gọi LLM mặc định để đảm bảo tốc độ và chi phí bằng 0."},
    {"key": "m20", "session_id": "vi-01", "timestamp": "2026-11-11T10:30:00Z", "tags": ["vietnamese", "search"],
     "content": "Bộ lọc tìm kiếm hỗ trợ giới hạn kết quả theo document_id, version và pipeline_run_id khi so sánh thí nghiệm RAG."},
    {"key": "m21", "session_id": "vi-02", "timestamp": "2026-11-18T14:10:00Z", "tags": ["vietnamese", "user-pref"],
     "content": "Người dùng ưu tiên kiến trúc filesystem-first để dễ backup dữ liệu gốc và chạy kiểm thử cục bộ không phụ thuộc Neo4j."},
    {"key": "m22", "session_id": "vi-02", "timestamp": "2026-11-19T16:00:00Z", "tags": ["vietnamese", "deploy"],
     "content": "Khi triển khai production, web bind cổng 127.0.0.1:4321 sau reverse proxy và không chạy docker compose down -v để tránh mất volume."},

    # --- Distractors / realistic noise ---
    {"key": "m23", "session_id": "misc-01", "timestamp": "2026-08-20T12:00:00Z", "tags": ["meeting"],
     "content": "Weekly standup reminder: update sprint board every Tuesday morning before 10:00 UTC."},
    {"key": "m24", "session_id": "misc-01", "timestamp": "2026-11-01T18:00:00Z", "tags": ["docs"],
     "content": "Updated README badges, contributor guidelines, and local development troubleshooting tips."},
]

BENCHMARK_QUERIES = [
    # 1. Factual recall
    {"category": "factual_recall", "query": "Which vector database stores rebuildable dense embeddings with cosine similarity?", "relevant": ["m01"]},
    {"category": "factual_recall", "query": "What is the immutable source of truth for chunks and source provenance?", "relevant": ["m02"]},
    {"category": "factual_recall", "query": "How does HybridMemory combine signals from dense, graph, BM25, and temporal?", "relevant": ["m06"]},
    {"category": "factual_recall", "query": "Which memory strategy requires zero LLM or embedding API token cost?", "relevant": ["m05"]},

    # 2. Multi-hop / relational graph queries
    {"category": "multi_hop_graph", "query": "OrionParser Docling OCR Vietnamese PDF", "relevant": ["m07", "m08"]},
    {"category": "multi_hop_graph", "query": "SentinelProxy Nginx mTLS FastAPI port 4321", "relevant": ["m09", "m10"]},
    {"category": "multi_hop_graph", "query": "Neo4j Redis GraphMemory enrichment worker", "relevant": ["m03", "m04"]},
    {"category": "multi_hop_graph", "query": "Who authored OrionParser and what library does it extend?", "relevant": ["m07", "m08"]},

    # 3. Temporal reasoning
    {"category": "temporal_reasoning", "query": "What milestone was finalized on 2026-10-28 for MCP tools?", "relevant": ["m15"], "temporal_bucket": "2026-10"},
    {"category": "temporal_reasoning", "query": "What evaluation metrics were added on 2026-11-05?", "relevant": ["m16"], "temporal_bucket": "2026-11"},
    {"category": "temporal_reasoning", "query": "What launched on 2026-11-25 in Memory Lab v0.2.0?", "relevant": ["m18"], "temporal_bucket": "2026-11-25"},
    {"category": "temporal_reasoning", "query": "Stage cache invalidation API added on 2026-11-12", "relevant": ["m17"], "temporal_bucket": "2026-11"},

    # 4. Exact identifiers & error codes
    {"category": "exact_identifier", "query": "ERR-QDR-4091 ns_9a4f2c dimension mismatch", "relevant": ["m11", "m12"]},
    {"category": "exact_identifier", "query": "sha-7175082 E2E_BASE_URL=http://localhost:4321", "relevant": ["m13"]},
    {"category": "exact_identifier", "query": "MAX_UPLOAD_BYTES=52428800 BUN_INSTALL Bun 1.4.4", "relevant": ["m14"]},
    {"category": "exact_identifier", "query": "POST /runs/{id}/invalidate stage cache", "relevant": ["m17"]},

    # 5. Bilingual Vietnamese / English (with & without diacritics)
    {"category": "bilingual_vi_en", "query": "trich xuat thuc the quan he moc thoi gian khong goi LLM", "relevant": ["m19"]},
    {"category": "bilingual_vi_en", "query": "Bộ lọc document_id version pipeline_run_id khi so sánh thí nghiệm RAG", "relevant": ["m20"]},
    {"category": "bilingual_vi_en", "query": "kien truc filesystem-first backup du lieu goc khong phu thuoc Neo4j", "relevant": ["m21"]},
    {"category": "bilingual_vi_en", "query": "triển khai production 127.0.0.1:4321 không chạy docker compose down -v", "relevant": ["m22"]},
]

dataset_hash = digest(
    json.dumps({"memories": BENCHMARK_MEMORIES, "queries": BENCHMARK_QUERIES}, sort_keys=True, ensure_ascii=False).encode()
)
print(f"Dataset ready: {len(BENCHMARK_MEMORIES)} memories, {len(BENCHMARK_QUERIES)} queries | hash={dataset_hash[:12]}")


## 4. Định nghĩa Adapter đồng nhất cho Memory Lab & các hệ thống đối chứng

Mỗi hệ thống được bọc bởi cùng một interface `BenchmarkAdapter`:
- `ingest(memories)` -> trả về thời gian nạp (ms) và số lần gọi embedding/LLM
- `search(query, k, temporal_bucket=None)` -> trả về danh sách `key` top-K được xếp hạng

Điều này đảm bảo so sánh **công bằng tuyệt đối** (cùng corpus, cùng embedding model, cùng K, cùng công thức `Recall@K`, `MRR`, `nDCG@K`).


In [ ]:
import shutil
import tempfile
import time
from pathlib import Path
import numpy as np
from qdrant_client import QdrantClient

from memory_lab.memory.service import MemoryService
from memory_lab.storage.filesystem.store import FileStore
from memory_lab.storage.qdrant.index import QdrantIndex


class BenchmarkAdapter:
    name: str
    family: str

    def ingest(self, memories: list[dict]) -> dict:
        raise NotImplementedError

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        raise NotImplementedError

    def close(self) -> None:
        pass


class MemoryLabAdapter(BenchmarkAdapter):
    """Benchmarks a specific Memory Lab strategy (zero_token, graph, vector, hybrid)."""

    def __init__(self, strategy: str, embedding_provider: CountingEmbeddingProvider):
        self.strategy = strategy
        self.name = f"MemoryLab-{strategy}"
        self.family = "Memory Lab"
        self.embedder = embedding_provider
        self._tmpdir = Path(tempfile.mkdtemp(prefix=f"mlab-{strategy}-"))
        self._qdrant = QdrantClient(":memory:")
        self._store = FileStore(self._tmpdir)
        self._index = QdrantIndex(self._qdrant, f"bench_{strategy}")
        self._service = MemoryService(
            self._store,
            self.embedder,
            self._index,
            self.embedder.identity,
            default_strategy=strategy,
        )
        self._id_to_key: dict[str, str] = {}

    def ingest(self, memories: list[dict]) -> dict:
        self.embedder.reset_counters()
        t0 = time.perf_counter()
        for item in memories:
            rec = self._service.store_memory(
                content=item["content"],
                session_id=item["session_id"],
                timestamp=item["timestamp"],
                tags=item.get("tags", []),
                strategy=self.strategy,
                metadata={"bench_key": item["key"]},
            )
            self._id_to_key[rec.id] = item["key"]
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        return {
            "ingest_ms": round(elapsed_ms, 2),
            "ingest_embed_calls": self.embedder.api_calls,
            "ingest_texts_embedded": self.embedder.texts_embedded,
        }

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        hits = self._service.search_memory(
            query, strategy=self.strategy, limit=k, temporal_bucket=temporal_bucket
        )
        return [self._id_to_key.get(hit.record.id, hit.record.id) for hit in hits]

    def close(self) -> None:
        self._qdrant.close()
        shutil.rmtree(self._tmpdir, ignore_errors=True)


class NaiveRecencyAdapter(BenchmarkAdapter):
    """Baseline: returns the K most recent memories (standard sliding-window context)."""

    name = "Naive-Recency"
    family = "Baseline"

    def __init__(self):
        self._items: list[dict] = []

    def ingest(self, memories: list[dict]) -> dict:
        t0 = time.perf_counter()
        self._items = sorted(memories, key=lambda x: x["timestamp"], reverse=True)
        return {"ingest_ms": round((time.perf_counter() - t0) * 1000.0, 2), "ingest_embed_calls": 0, "ingest_texts_embedded": 0}

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        return [item["key"] for item in self._items[:k]]


class RankBM25Adapter(BenchmarkAdapter):
    """External Baseline: standard rank-bm25 Okapi BM25 lexical retrieval."""

    name = "RankBM25"
    family = "External"

    def __init__(self):
        self._bm25 = None
        self._keys: list[str] = []

    def ingest(self, memories: list[dict]) -> dict:
        from rank_bm25 import BM25Okapi
        t0 = time.perf_counter()
        self._keys = [m["key"] for m in memories]
        corpus = [m["content"].lower().split() for m in memories]
        self._bm25 = BM25Okapi(corpus)
        return {"ingest_ms": round((time.perf_counter() - t0) * 1000.0, 2), "ingest_embed_calls": 0, "ingest_texts_embedded": 0}

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        scores = self._bm25.get_scores(query.lower().split())
        order = np.argsort(scores)[::-1][:k]
        return [self._keys[i] for i in order if scores[i] > 0]


class FAISSAdapter(BenchmarkAdapter):
    """External Baseline: FAISS IndexFlatIP exact cosine similarity vector memory."""

    name = "FAISS-FlatIP"
    family = "External"

    def __init__(self, embedding_provider: CountingEmbeddingProvider):
        self.embedder = embedding_provider
        self._index = None
        self._keys: list[str] = []

    def ingest(self, memories: list[dict]) -> dict:
        import faiss
        self.embedder.reset_counters()
        t0 = time.perf_counter()
        self._keys = [m["key"] for m in memories]
        vecs = np.array(self.embedder.embed([m["content"] for m in memories]), dtype="float32")
        faiss.normalize_L2(vecs)
        self._index = faiss.IndexFlatIP(vecs.shape[1])
        self._index.add(vecs)
        return {
            "ingest_ms": round((time.perf_counter() - t0) * 1000.0, 2),
            "ingest_embed_calls": self.embedder.api_calls,
            "ingest_texts_embedded": self.embedder.texts_embedded,
        }

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        import faiss
        q = np.array(self.embedder.embed([query]), dtype="float32")
        faiss.normalize_L2(q)
        _scores, idxs = self._index.search(q, k)
        return [self._keys[i] for i in idxs[0] if 0 <= i < len(self._keys)]


class ChromaDBAdapter(BenchmarkAdapter):
    """External Baseline: ChromaDB in-memory dense vector memory."""

    name = "ChromaDB-Dense"
    family = "External"

    def __init__(self, embedding_provider: CountingEmbeddingProvider):
        self.embedder = embedding_provider
        self._client = None
        self._collection = None

    def ingest(self, memories: list[dict]) -> dict:
        import chromadb
        self.embedder.reset_counters()
        t0 = time.perf_counter()
        self._client = chromadb.Client()
        col_name = f"bench_{int(time.time() * 1000)}"
        self._collection = self._client.create_collection(name=col_name, metadata={"hnsw:space": "cosine"})
        embeddings = self.embedder.embed([m["content"] for m in memories])
        self._collection.add(
            ids=[m["key"] for m in memories],
            documents=[m["content"] for m in memories],
            embeddings=embeddings,
            metadatas=[{"session_id": m["session_id"], "timestamp": m["timestamp"]} for m in memories],
        )
        return {
            "ingest_ms": round((time.perf_counter() - t0) * 1000.0, 2),
            "ingest_embed_calls": self.embedder.api_calls,
            "ingest_texts_embedded": self.embedder.texts_embedded,
        }

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        q_emb = self.embedder.embed([query])
        res = self._collection.query(query_embeddings=q_emb, n_results=k)
        return list(res["ids"][0]) if res.get("ids") else []


class Mem0OptionalAdapter(BenchmarkAdapter):
    """Optional External System: Mem0 OSS (requires ENABLE_MEM0=True and OPENAI_API_KEY)."""

    name = "Mem0-OSS"
    family = "External (LLM)"

    def __init__(self):
        from mem0 import Memory
        self._mem = Memory()
        self._user_id = f"bench-{int(time.time())}"
        self._text_to_key: dict[str, str] = {}

    def ingest(self, memories: list[dict]) -> dict:
        t0 = time.perf_counter()
        for item in memories:
            self._mem.add(item["content"], user_id=self._user_id, metadata={"bench_key": item["key"]})
            self._text_to_key[item["content"]] = item["key"]
        return {"ingest_ms": round((time.perf_counter() - t0) * 1000.0, 2), "ingest_embed_calls": len(memories), "ingest_texts_embedded": len(memories)}

    def search(self, query: str, k: int, temporal_bucket: str | None = None) -> list[str]:
        res = self._mem.search(query, user_id=self._user_id, limit=k)
        items = res.get("results", res) if isinstance(res, dict) else res
        keys = []
        for entry in items[:k]:
            meta = entry.get("metadata") or {}
            if "bench_key" in meta:
                keys.append(meta["bench_key"])
        return keys

print("Adapters defined successfully.")


## 5. Chạy Benchmark & Thu thập Metrics (`Recall@K`, `MRR`, `nDCG@K`, `Latency`, `Cost`)


In [ ]:
import pandas as pd
from memory_lab.experiments.metrics import mean_reciprocal_rank, ndcg_at_k, recall_at_k


def build_adapters() -> list[BenchmarkAdapter]:
    adapters: list[BenchmarkAdapter] = [
        MemoryLabAdapter("zero_token", embedder),
        MemoryLabAdapter("graph", embedder),
        MemoryLabAdapter("vector", embedder),
        MemoryLabAdapter("hybrid", embedder),
        NaiveRecencyAdapter(),
    ]
    # Add optional external libraries gracefully if installed
    try:
        import rank_bm25  # noqa: F401
        adapters.append(RankBM25Adapter())
    except ImportError:
        print("Skipping RankBM25 (install rank-bm25)")

    try:
        import faiss  # noqa: F401
        adapters.append(FAISSAdapter(embedder))
    except ImportError:
        print("Skipping FAISS-FlatIP (install faiss-cpu)")

    try:
        import chromadb  # noqa: F401
        adapters.append(ChromaDBAdapter(embedder))
    except ImportError:
        print("Skipping ChromaDB-Dense (install chromadb)")

    if ENABLE_MEM0 and os.environ.get("OPENAI_API_KEY"):
        adapters.append(Mem0OptionalAdapter())
    elif ENABLE_MEM0:
        print("Skipping Mem0-OSS: set os.environ['OPENAI_API_KEY'] to enable")

    return adapters


def run_full_benchmark(
    adapters: list[BenchmarkAdapter],
    memories: list[dict],
    queries: list[dict],
    k: int = 5,
    use_temporal_hint: bool = True,
):
    summary_rows = []
    per_query_rows = []

    for adapter in adapters:
        print(f"Benchmarking {adapter.name}...")
        ingest_stats = adapter.ingest(memories)

        embedder.reset_counters()
        latencies = []
        r1_list, r3_list, r5_list, mrr_list, ndcg_list = [], [], [], [], []

        for q in queries:
            bucket = q.get("temporal_bucket") if use_temporal_hint else None
            t0 = time.perf_counter()
            retrieved = adapter.search(q["query"], k=k, temporal_bucket=bucket)
            latency_ms = (time.perf_counter() - t0) * 1000.0
            latencies.append(latency_ms)

            rel_set = set(q["relevant"])
            flags = [item_id in rel_set for item_id in retrieved]
            total_rel = len(rel_set)

            r1 = recall_at_k(flags, total_rel, 1)
            r3 = recall_at_k(flags, total_rel, 3)
            rk = recall_at_k(flags, total_rel, k)
            mrr = mean_reciprocal_rank(flags, k)
            ndcg = ndcg_at_k(flags, total_rel, k)

            r1_list.append(r1)
            r3_list.append(r3)
            r5_list.append(rk)
            mrr_list.append(mrr)
            ndcg_list.append(ndcg)

            per_query_rows.append(
                {
                    "system": adapter.name,
                    "family": adapter.family,
                    "category": q["category"],
                    "query": q["query"],
                    "expected": q["relevant"],
                    "retrieved": retrieved,
                    "recall@1": r1,
                    "recall@3": r3,
                    f"recall@{k}": rk,
                    f"mrr@{k}": mrr,
                    f"ndcg@{k}": ndcg,
                    "latency_ms": round(latency_ms, 3),
                }
            )

        summary_rows.append(
            {
                "System": adapter.name,
                "Family": adapter.family,
                "Recall@1": round(float(np.mean(r1_list)), 4),
                "Recall@3": round(float(np.mean(r3_list)), 4),
                f"Recall@{k}": round(float(np.mean(r5_list)), 4),
                f"MRR@{k}": round(float(np.mean(mrr_list)), 4),
                f"nDCG@{k}": round(float(np.mean(ndcg_list)), 4),
                "Query p50 (ms)": round(float(np.percentile(latencies, 50)), 2),
                "Query p95 (ms)": round(float(np.percentile(latencies, 95)), 2),
                "Ingest (ms)": ingest_stats["ingest_ms"],
                "Embed Calls (Ingest+Query)": ingest_stats["ingest_embed_calls"] + embedder.api_calls,
            }
        )
        adapter.close()

    summary_df = pd.DataFrame(summary_rows).sort_values(by=f"nDCG@{k}", ascending=False).reset_index(drop=True)
    detail_df = pd.DataFrame(per_query_rows)
    return summary_df, detail_df


adapters = build_adapters()
summary_df, detail_df = run_full_benchmark(adapters, BENCHMARK_MEMORIES, BENCHMARK_QUERIES, k=TOP_K)
display(summary_df)


## 6. Biểu đồ so sánh & Phân tích theo từng nhóm câu hỏi


In [ ]:
import matplotlib.pyplot as plt

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# 1. Grouped Bar Chart: Recall@K, MRR@K, nDCG@K
metrics_cols = [f"Recall@{TOP_K}", f"MRR@{TOP_K}", f"nDCG@{TOP_K}"]
x = np.arange(len(summary_df))
width = 0.25
colors = ["#6ee7b7", "#60a5fa", "#fbbf24"]
for idx, col in enumerate(metrics_cols):
    axes[0].bar(x + (idx - 1) * width, summary_df[col], width=width, label=col, color=colors[idx])
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["System"], rotation=30, ha="right", fontsize=9)
axes[0].set_ylim(0, 1.08)
axes[0].set_ylabel("Score")
axes[0].set_title(f"Retrieval Quality (K={TOP_K})")
axes[0].legend(loc="lower right")
axes[0].grid(axis="y", alpha=0.2)

# 2. Quality vs Latency Pareto Scatter
for _, row in summary_df.iterrows():
    color = "#34d399" if row["Family"] == "Memory Lab" else "#94a3b8"
    marker = "o" if row["Embed Calls (Ingest+Query)"] > 0 else "D"
    axes[1].scatter(row["Query p50 (ms)"], row[f"nDCG@{TOP_K}"], s=140, c=color, marker=marker, edgecolors="white", linewidth=0.8)
    axes[1].annotate(row["System"], (row["Query p50 (ms)"], row[f"nDCG@{TOP_K}"]), xytext=(6, 4), textcoords="offset points", fontsize=8.5)
axes[1].set_xlabel("Query p50 Latency (ms) — lower is faster")
axes[1].set_ylabel(f"nDCG@{TOP_K} — higher is better")
axes[1].set_title("Quality vs Speed (Diamond = 0 Embed Calls)")
axes[1].grid(alpha=0.2)

# 3. Category Breakdown Heatmap (nDCG@K)
pivot = detail_df.pivot_table(index="system", columns="category", values=f"ndcg@{TOP_K}", aggfunc="mean")
pivot = pivot.reindex(summary_df["System"])
im = axes[2].imshow(pivot.values, cmap="YlGn", vmin=0.0, vmax=1.0, aspect="auto")
axes[2].set_xticks(np.arange(len(pivot.columns)))
axes[2].set_xticklabels(pivot.columns, rotation=30, ha="right", fontsize=9)
axes[2].set_yticks(np.arange(len(pivot.index)))
axes[2].set_yticklabels(pivot.index, fontsize=9)
axes[2].set_title(f"nDCG@{TOP_K} by Query Category")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        axes[2].text(j, i, f"{val:.2f}", ha="center", va="center", color="black" if val > 0.55 else "white", fontsize=8.5, fontweight="bold")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("memory_lab_benchmark_charts.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Kiểm chứng độ tái lập với `ExperimentService` tích hợp sẵn của Memory Lab & Xuất báo cáo

Cell này chạy lại đánh giá bằng chính [`ExperimentService`](https://github.com/npv2k1/memory-lab) của Memory Lab để lưu `dataset_hash`, `config_snapshot` và xuất báo cáo JSON + CSV để lưu trữ hoặc đối chiếu sau này.


In [ ]:
from memory_lab.core.models import BenchmarkCase, BenchmarkDatasetCreate, ExperimentRunRequest
from memory_lab.documents.pipeline.service import DocumentPipeline
from memory_lab.experiments.service import ExperimentService

export_dir = Path("benchmark_outputs")
export_dir.mkdir(exist_ok=True)

summary_df.to_csv(export_dir / "summary.csv", index=False)
detail_df.to_csv(export_dir / "per_query_details.csv", index=False)

# Run native Memory Lab ExperimentService comparison across its 4 strategies
tmp_exp = Path(tempfile.mkdtemp(prefix="mlab-exp-"))
qclient = QdrantClient(":memory:")
fstore = FileStore(tmp_exp)
qindex = QdrantIndex(qclient, "native_exp")
pipe = DocumentPipeline(fstore, embedder, qindex, embedder.identity)
mem_svc = MemoryService(fstore, embedder, qindex, embedder.identity)
exp_svc = ExperimentService(fstore, pipe, mem_svc)

key_to_id = {}
for item in BENCHMARK_MEMORIES:
    rec = mem_svc.store_memory(
        content=item["content"],
        session_id=item["session_id"],
        timestamp=item["timestamp"],
        tags=item["tags"],
        strategy="hybrid",
    )
    key_to_id[item["key"]] = rec.id

ds = exp_svc.create_dataset(
    BenchmarkDatasetCreate(
        name="colab-agent-memory-benchmark",
        description="Native Memory Lab reproducible snapshot",
        cases=[
            BenchmarkCase(query=q["query"], relevant_memory_ids=[key_to_id[k] for k in q["relevant"]])
            for q in BENCHMARK_QUERIES
        ],
    )
)

report_ids = []
for strat in ("zero_token", "graph", "vector", "hybrid"):
    rep = exp_svc.run_experiment(
        ExperimentRunRequest(
            name=f"memory-lab-{strat}",
            dataset_id=ds.id,
            target="memory",
            memory_strategy=strat,
            k=TOP_K,
        )
    )
    report_ids.append(rep.id)

comparison = exp_svc.compare_reports(report_ids)
(export_dir / "memory_lab_comparison.json").write_text(json.dumps(comparison, indent=2, ensure_ascii=False))
qclient.close()
shutil.rmtree(tmp_exp, ignore_errors=True)

print("Saved outputs to:", export_dir.resolve())
print("Native dataset_hash:", ds.dataset_hash)
for row in comparison["reports"]:
    print(f"  {row['name']:22s} | Recall@{row['k']}={row['metrics']['recall_at_k']:.3f} | MRR={row['metrics']['mrr']:.3f} | nDCG={row['metrics']['ndcg_at_k']:.3f}")


## 8. (Tùy chọn) Nạp tập dữ liệu tùy chỉnh (LoCoMo / LongMemEval / JSON)

Bạn có thể upload file JSON có dạng `{"memories": [...], "queries": [...]}` trên Colab để benchmark hệ thống trên dữ liệu thật của riêng bạn.


In [ ]:
#@title Upload & Benchmark Custom JSON Dataset (Optional)
RUN_CUSTOM_DATASET = False  #@param {type:"boolean"}

if RUN_CUSTOM_DATASET:
    try:
        from google.colab import files
        uploaded = files.upload()
        filename = next(iter(uploaded.keys()))
        payload = json.loads(uploaded[filename].decode("utf-8"))
        custom_summary, custom_detail = run_full_benchmark(
            build_adapters(), payload["memories"], payload["queries"], k=TOP_K
        )
        display(custom_summary)
    except ImportError:
        print("google.colab.files is only available inside Google Colab.")
else:
    print("Set RUN_CUSTOM_DATASET = True to upload your own benchmark JSON.")
